# upload
`pairs.jsonl` -> HuggingFace (seeded split + push + card). Needs HF auth.

In [ ]:
import json, os
from pathlib import Path
from datasets import Dataset, DatasetDict
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))
SPLIT = os.environ["SPLIT"]
ROOT  = Path("../../data")
PAIRS = ROOT / "output" / SPLIT / "pairs.jsonl"

HF_REPO   = "notoh/exebench_llvm_o3"
PRIVATE   = True
TEST_FRAC = 0.1
SEED      = 42   # frozen split -> every model size sees identical train/test

In [ ]:
# === load local pairs, make a seeded split, push to HF ===
# auth first (terminal):  hf auth login   (or export HF_TOKEN=hf_...)
rows = [json.loads(l) for l in PAIRS.open() if l.strip()]
print(f"{len(rows)} pairs loaded from {PAIRS}")

ds = Dataset.from_list(rows)
sp = ds.train_test_split(test_size=TEST_FRAC, seed=SEED)
dd = DatasetDict({"train": sp["train"], "test": sp["test"]})
print(dd)

commit_msg = f"{len(rows)} pairs from {SPLIT} ({len(sp['train'])} train / {len(sp['test'])} test)"
info = dd.push_to_hub(HF_REPO, private=PRIVATE, commit_message=commit_msg)
print("pushed. RECORD THE COMMIT SHA -> pin it in training:")
print(info)

In [ ]:
# === confirm what is ON HF matches local (load from the pinned revision) ===
REVISION = os.environ["REVISION"]
from datasets import load_dataset
chk = load_dataset(HF_REPO, revision=REVISION, split="train")
r = chk[0]
print("loaded", len(chk), "train rows from", REVISION)
print("c_source clean (no # lines):", not any(l.lstrip().startswith("#") for l in r["c_source"].splitlines()))
print("o3_ir clean (no attributes/!llvm):", "\nattributes #" not in r["o3_ir"] and "\n!llvm" not in r["o3_ir"])
print("o3_ir has define:", "define " in r["o3_ir"])